# Train a simple regression model

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm, trange
import gensim.downloader as api
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

## Load the Data

In [ ]:
cds_file = 'CDSRatSecGeo.csv'
cds_data = pd.read_csv(cds_file, index_col=0)

In [ ]:
cds_data.head()

In [ ]:
# Group the DataFrame by the 'Category' column
grouped = cds_data.groupby('Date')

# Create a list of smaller DataFrames
dfs_by_date = [grouped.get_group(category) for category in grouped.groups]

In [ ]:
for df in dfs_by_date:
    df.reset_index(inplace=True)
    df.drop(['Date', 'Company', 'Ticker'], axis=1, inplace=True)

In [ ]:
max_names = 0
max_idx = None
for idx, df in enumerate(dfs_by_date):
    if len(df) > max_names:
        max_names = len(df)
        max_idx = idx

In [ ]:
max_idx

## Prepare the Data

In [ ]:
target = dfs_by_date[0][['PX1', 'PX2', 'PX3', 'PX4', 'PX5',
                         'PX6', 'PX7', 'PX8', 'PX9', 'PX10']]
input_features = dfs_by_date[0][['Sector', 'Country', 'Rating']]

In [ ]:
pd.set_option('display.max_rows', input_features.shape[0]+1)

In [ ]:
input_features.head(302)

In [ ]:
input_features.replace('United States', 'USA', inplace=True)
input_features.replace('United Kingdom', 'UK', inplace=True)
input_features.replace('South Korea', 'Korea', inplace=True)

In [ ]:
dfsec = pd.DataFrame()
dfsec['sector_encoded'] = pd.Categorical(input_features['Sector']).codes
category_array = np.array(dfsec['sector_encoded'])
num_classes = len(np.unique(category_array))
one_hot = np.eye(num_classes)[category_array]

In [ ]:
input_features['Sector'].unique()

In [ ]:
# Load pre-trained Word2Vec embeddings
word2vec_model = api.load('word2vec-google-news-300')

In [ ]:
embedding1 = word2vec_model[input_features['Country']]

In [ ]:
tsne = TSNE(n_components=2, random_state=0)  # n_components can be 2 or 3
reduced_embeddings = tsne.fit_transform(embedding1)

In [ ]:
unique_countries = input_features['Country'].unique()
first_occurrences = {country: input_features[input_features['Country'] == country].index[0] for country in unique_countries}

In [ ]:
country_data_dict = {country: reduced_embeddings[index] for country, index in first_occurrences.items()}

In [ ]:
country_data_dict

In [ ]:
input_features['Rating'].unique()

In [ ]:
rating_map = {'AAA':0, 'AA+':1, 'AA':2, 'AA-':3, 'A+': 4, 'A': 5, 'A-': 6, 
              'BBB+': 7, 'BBB': 8, 'BBB-': 9, 'BB+': 10,  'BB': 11, 'BB-': 12, 'B+': 13, 'B': 14, 'B-': 15}

In [ ]:
index_rating = input_features['Rating'].map(rating_map).to_numpy()

In [ ]:
index_rating = index_rating.reshape(len(index_rating), 1)

In [ ]:
input_tensor = np.concatenate([reduced_embeddings, one_hot, index_rating], axis=1)

In [ ]:
input_tensor

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(input_tensor, target.to_numpy(), test_size=0.1) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
batch_size = 16

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
epochs = 200
lr = 0.001

In [ ]:
# Swish activation function
class Swish(nn.Module):
    def forward(self, input):
        return input * torch.sigmoid(input)
    
# Feed-forward network
class FeedForwardNetwork(nn.Module):
    def __init__(self, input_size, num_hidden_layers, neurons_per_layer, output_size):
        super(FeedForwardNetwork, self).__init__()

        # Construct layers
        layers = []

        # Input layer
        layers.append(nn.Linear(input_size, neurons_per_layer))
        layers.append(Swish())

        # Hidden layers
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(Swish())

        # Output layer
        layers.append(nn.Linear(neurons_per_layer, output_size))

        # Combine layers
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
model = FeedForwardNetwork(14, 1, 16, 10).to(device)

## Build and Train the model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = F.mse_loss(outputs.squeeze(), batch_y)  # Assuming MSE is the desired error metric
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train error: {train_loss:.4f}, Test error: {test_loss:.4f}")

    return train_errors, test_errors

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
model_out = train_model(model, data_loader, test_data_loader, loss_fn, optimizer, epochs)

In [ ]:
reduced_embeddings[0] # USA

In [ ]:
one_hot_ex = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.float32)

In [ ]:
rating_ex = np.array([4.0])

In [ ]:
rating_ex

In [ ]:
input_ex = np.expand_dims(np.concatenate([reduced_embeddings[0], one_hot_ex, rating_ex]), axis=0)

In [ ]:
input_ex_sc = scaler.transform(input_ex)

In [ ]:
tinput_ex_sc = torch.tensor(input_ex_sc).float().to(device)

In [ ]:
ex_res = model(tinput_ex_sc)

In [ ]:
ex_res

In [ ]:
target

In [ ]:
dfsec['sector_encoded'] 

## Testing and Analysis

Build curves for USA for one sector for all ratings

In [ ]:
usa = country_data_dict['USA']

In [ ]:
one_hot_ex = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], dtype=np.float32)

In [ ]:
usa_all = np.tile(usa, (16, 1))

In [ ]:
one_hot_ex_all = np.tile(one_hot_ex, (16, 1))

In [ ]:
rating_lst = [i for i in range(16)]
rating_arr = np.array(rating_lst).reshape(-1, 1)

In [ ]:
input_tensor_ex = np.concatenate([usa_all, one_hot_ex_all, rating_arr], axis=1)

In [ ]:
input_ex_sc = scaler.transform(input_tensor_ex)
tinput_ex_sc = torch.tensor(input_ex_sc).float().to(device)

In [ ]:
ex_res = model(tinput_ex_sc)

In [ ]:
ex_res

In [ ]:
ex_res_np = numpy_array = ex_res.detach().cpu().numpy()

In [ ]:
ex_res_df = pd.DataFrame(ex_res_np)

In [ ]:
ex_res_df.to_latex()

Build curves for Japan and Financial

In [ ]:
japan = country_data_dict['Japan']

one_hot_ex = np.array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], dtype=np.float32)

japan_all = np.tile(japan, (16, 1))

one_hot_ex_all = np.tile(one_hot_ex, (16, 1))

rating_lst = [i for i in range(16)]
rating_arr = np.array(rating_lst).reshape(-1, 1)

input_tensor_ex = np.concatenate([japan_all, one_hot_ex_all, rating_arr], axis=1)

input_ex_sc = scaler.transform(input_tensor_ex)
tinput_ex_sc = torch.tensor(input_ex_sc).float().to(device)

ex_res = model(tinput_ex_sc)

In [ ]:
ex_res_np = numpy_array = ex_res.detach().cpu().numpy()
ex_res_df = pd.DataFrame(ex_res_np)
ex_res_df.to_latex()